# Evaluating a Chatbot Conversation with DeepEval

Everything so far scored **one** input/output pair. A chatbot has memory, a role to stay in, and a
goal that spans many turns — a single `LLMTestCase` can't capture that. DeepEval instead uses a
**`ConversationalTestCase`**: a list of `Turn`s, plus some context about the conversation itself
(`scenario`, `chatbot_role`, `expected_outcome`).

| Metric | What it checks |
| --- | --- |
| `RoleAdherenceMetric` | Did the assistant stay in its assigned role across every turn? |
| `ConversationCompletenessMetric` | Did the conversation fulfil the user's high-level intentions? |
| `TurnRelevancyMetric` | Is each assistant turn relevant given the conversation so far? |
| `KnowledgeRetentionMetric` | Did the assistant remember facts the user already gave it? |
| `GoalAccuracyMetric` | Across the turns, did the assistant achieve the user's underlying goal? |
| `TopicAdherenceMetric` | Did the conversation stay within allowed topics? |
| `ConversationalGEval` | Your own multi-turn criteria (the conversational version of `GEval`). |

⚠️ Don't mix families: single-turn metrics like `AnswerRelevancyMetric` cannot score a
`ConversationalTestCase`, and conversational metrics cannot score a plain `LLMTestCase`.

## 1. Install packages

We only need three libraries: `deepeval` itself, `google-genai` (so DeepEval's judge can talk to
Gemini), and `groq` (the model we're evaluating). No OpenAI key anywhere in this notebook.

In [1]:
%pip install -q deepeval google-genai groq


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 2. Add your API keys

Two roles, kept separate the whole way through this course:

- **Groq** — the *model under test*. It free-tier serves `llama-3.3-70b-versatile`. Get a key at
  https://console.groq.com/keys
- **Gemini** — the *judge*. Every DeepEval metric needs an LLM to grade with, and we use Gemini so
  you never need an OpenAI key. Get a free key at https://aistudio.google.com/apikey

Keys are entered with `getpass` so they never get printed or saved into the notebook file.

In [2]:
import os, getpass

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")  # skip DeepEval's anonymous telemetry

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your GEMINI_API_KEY: ")
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]  # some libs read this name instead

print("Keys set.")

Keys set.


## 3. Build a short conversation by hand

Normally these turns come from real chat logs, or from a **simulator** (next notebook). Here we
write them out so we can focus purely on the evaluation code. The user asks a real question, gets
a good answer, then goes off-topic — a nice test for `TopicAdherenceMetric` and `RoleAdherenceMetric`.

In [3]:
from deepeval.test_case import Turn

turns = [
    Turn(role="user", content="Hey, what's a context window in an LLM?"),
    Turn(role="assistant", content=(
        "The context window is the maximum number of tokens the model can consider at once -- "
        "covering the system prompt, the conversation history, and its own reply.")),
    Turn(role="user", content="Does a bigger context window always mean better answers?"),
    Turn(role="assistant", content=(
        "Not necessarily. A bigger window gives more room for context, but the model still has to "
        "use it well -- very long contexts can dilute attention, so retrieval quality matters as "
        "much as sheer size.")),
    Turn(role="user", content="Cool, unrelated, but do you know a good pizza place nearby?"),
    Turn(role="assistant", content=(
        "I'm a GenAI concepts assistant, so I can't help with restaurant recommendations -- happy "
        "to keep answering questions about LLMs and RAG though!")),
]

## 4. Wrap it in a `ConversationalTestCase`

`chatbot_role` and `scenario` give the judge context it can't infer from the turns alone.

In [4]:
from deepeval.test_case import ConversationalTestCase

test_case = ConversationalTestCase(
    turns=turns,
    chatbot_role="a GenAI concepts assistant that explains LLM and RAG topics, and declines "
                  "unrelated requests",
    scenario="A developer is casually asking a documentation chatbot about context windows.",
    expected_outcome="The user understands what a context window is and its trade-offs.",
)

## The judge: `GeminiModel`

Every DeepEval metric defaults to OpenAI if you don't tell it otherwise — even a metric that
scores deterministically will still try to build an OpenAI client unless you pass `model=`. We
build **one Gemini judge** here and pass it to every metric in this notebook.

In [5]:
from deepeval.models import GeminiModel

judge = GeminiModel(model="gemini-2.5-flash", api_key=os.environ["GEMINI_API_KEY"], temperature=0)
print("Judge ready:", judge.get_model_name())

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Judge ready: gemini-2.5-flash (Gemini)


## 5. Define conversational metrics

`TopicAdherenceMetric` needs a constructor argument, `relevant_topics` — the only metric here that
does. `ConversationalGEval` is our own custom criterion, the multi-turn sibling of `GEval`.

In [6]:
from deepeval.metrics import (
    ConversationalGEval,
    ConversationCompletenessMetric,
    GoalAccuracyMetric,
    KnowledgeRetentionMetric,
    RoleAdherenceMetric,
    TopicAdherenceMetric,
    TurnRelevancyMetric,
)
from deepeval.test_case.conversational_test_case import MultiTurnParams

metrics = [
    RoleAdherenceMetric(model=judge),
    ConversationCompletenessMetric(model=judge),
    TurnRelevancyMetric(model=judge),
    KnowledgeRetentionMetric(model=judge),
    GoalAccuracyMetric(model=judge),
    TopicAdherenceMetric(relevant_topics=["large language models", "RAG", "AI agents", "GenAI concepts"], model=judge),
    ConversationalGEval(
        name="Helpfulness",
        criteria="Across the conversation, did the assistant give technically accurate, easy to "
                 "understand explanations, and handle the off-topic request gracefully?",
        evaluation_params=[MultiTurnParams.ROLE, MultiTurnParams.CONTENT],
        model=judge,
    ),
]

## 6. Run the evaluation

In [7]:
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig, ErrorConfig

results = evaluate(
    test_cases=[test_case],
    metrics=metrics,
    async_config=AsyncConfig(max_concurrent=2),
    display_config=DisplayConfig(print_results=False),
    error_config=ErrorConfig(ignore_errors=True),
)

✨ You're running DeepEval's latest Role Adherence Metric! (using gemini-2.5-flash (Gemini), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Conversation Completeness Metric! (using gemini-2.5-flash (Gemini), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Turn Relevancy Metric! (using gemini-2.5-flash (Gemini), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Knowledge Retention Metric! (using gemini-2.5-flash (Gemini), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Goal Accuracy Metric! (using gemini-2.5-flash (Gemini), strict=False, 
async_mode=True)...

...

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x105cd4840> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x105cd4840> is already entered

⚠ WARNING: All metrics errored across every test case — no metric scores were recorded. Posting the run anyway so 
you can inspect the trace + span errors on the Confident AI dashboard.

⚠ WARNING: No hyperparameters logged.
» ]8;id=938608;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.12s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## 7. Read the results

Conversational `TestResult`s expose `.turns` instead of a single `.input` / `.actual_output`.

In [8]:
for tr in results.test_results:
    print("=" * 90)
    print(f"Conversation with {len(tr.turns)} turns")
    for md_ in tr.metrics_data:
        verdict = "PASS" if md_.success else "FAIL"
        score = f"{md_.score:.2f}" if md_.score is not None else "ERRORED"
        print(f"  [{verdict}] {md_.name}: {score}")
        print(f"      reason: {md_.reason or md_.error}")

Conversation with 6 turns
  [FAIL] Role Adherence: ERRORED
      reason: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 388.839504ms.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimens

## Takeaway

`RoleAdherenceMetric` and `TopicAdherenceMetric` should both score high here — the assistant
declined the pizza question instead of answering it, which is correct role behaviour, not a
failure. If you rewrote turn 6 to actually recommend a pizza place, watch those two scores drop
while `ConversationCompletenessMetric` might barely move — that's the point of having several
metrics: each one is blind to failures outside its own lane.

Next: [`04_synthetic_golden_data_generation.ipynb`](04_synthetic_golden_data_generation.ipynb) —
stop hand-writing eval data.